# 吸菸行為偵測 — Google Colab

**功能：** YOLOv8-Pose 姿態估計 + 香菸偵測 + 煙霧偵測（條件三）  
**開始前：** 上方選單 → `執行階段 → 變更執行階段類型 → T4 GPU`

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. 下載專案

In [ ]:
%cd /content
!rm -rf cgr_detection
!git clone -b claude/funny-carson-pru37g https://github.com/tghtaipei/cgr_detection.git
%cd cgr_detection
!echo "✅ 專案已下載：$(pwd)"

## 2. 安裝套件

In [ ]:
!pip install -q openvino==2024.6.0 ultralytics onnxruntime opencv-python-headless numpy
print("✅ 套件安裝完成")

## 3. 上傳影片

In [ ]:
from google.colab import files
import shutil, os

uploaded = files.upload()
video_name = list(uploaded.keys())[0]
dest = f"/content/{video_name}"
if not os.path.exists(dest):
    shutil.move(video_name, dest)
print(f"✅ 已上傳：{dest}，大小：{os.path.getsize(dest)/1e6:.1f} MB")

## 4. 執行偵測

| 參數 | 說明 | 預設 |
|------|------|------|
| `cgr_conf` | 香菸偵測信心閾值 | 0.4 |
| `cig_box` | 顯示香菸框 | True |
| `skeleton` | 顯示骨架關節 | False |
| `threshold` | 連續幀吸菸判定閾值 | 50 |

In [ ]:
import subprocess

result = subprocess.run(
    ['python', 'infer_colab.py',
     '--source', f'/content/{video_name}',
     '--output', '/content/output.avi',
     '--cgr_conf', '0.4',
     '--cig_box'],
    capture_output=True, text=True,
    cwd='/content/cgr_detection'
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 5. 預覽結果

In [ ]:
import os

if not os.path.exists('/content/output.avi'):
    raise FileNotFoundError("找不到 output.avi，請確認第 4 格執行成功")

!ffmpeg -i /content/output.avi -vcodec libx264 -acodec aac -y /content/preview.mp4 -loglevel quiet
print("✅ 預覽檔已產生：/content/preview.mp4")

from IPython.display import HTML
from base64 import b64encode

mp4 = open('/content/preview.mp4', 'rb').read()
data = b64encode(mp4).decode()
HTML(f'<video controls width="800"><source src="data:video/mp4;base64,{data}"></video>')

## 6. 下載結果影片

In [ ]:
from google.colab import files
files.download('/content/output.avi')
print("下載完成")